In [ ]:
def eeg_classifier_cv(time, data,train_index, test_index):
    
    # Define X, y
    X = data.drop(columns=['Group']).values
    y = data['Group'].values
    
    # Run classifier model by using cross-validation, feature-selection and storing results.
    results = []

    for clf_name, clf in classifiers.items():
        print('Running:', clf_name)
        for selector_nm, selector in feature_selector.items():
            accuracy_scores = []
            f1_scores = []
            auc_scores = []

            vote_accuracy_scores = []
            vote_f1_scores = []
            vote_auc_scores = []

            sen_scores = []
            spe_scores = []
            ppv_scores = []
            npv_scores = []                

            vote_sen_scores = []
            vote_spe_scores = []
            vote_ppv_scores = []
            vote_npv_scores = []                

            vote_pred_list = []
            vote_test_list = []
            pred_list = []
            test_list = []

            X_train_num, X_test_num = train_index, test_index

            X_train = []
            y_train = []
            vote_num = []
            for X_tr in X_train_num:
                if X_tr < 36:
                    tr = np.load("D:/test5/raw"+ str(X_tr+1) +"_AD_"+time+".npy")
                    y_train += [1] * tr.shape[0]
                elif X_tr > 35 and X_tr < 66:
                    tr = np.load("D:/test5/raw"+ str(X_tr) +"_CN_"+time+".npy") 
                    y_train += [0] * tr.shape[0]
                elif X_tr > 65:
                    tr = np.load("D:/test5/raw"+ str(X_tr) +"_FTD_"+time+".npy")
                    y_train += [1] * tr.shape[0]
                else:
                    print('none',X_tr)
                X_train.append(tr)
            X_train = np.concatenate(X_train, axis=0)

            X_test = []
            y_test = []
            for X_te in X_test_num:
                if X_te < 36:
                    te = np.load("D:/test5/raw" + str(X_te+1) + "_AD_" +time+ ".npy")
                    y_test += [1] * te.shape[0]
                    vote_num.append(te.shape[0])
                elif X_te > 35 and X_te < 66:
                    te = np.load("D:/test5/raw"+ str(X_te) +"_CN_"+time+".npy") 
                    y_test += [0] * te.shape[0]
                    vote_num.append(te.shape[0])
                elif X_te > 65:
                    te = np.load("D:/test5/raw"+ str(X_te) +"_FTD_"+time+".npy")
                    
                    y_test += [1] * te.shape[0]
                    vote_num.append(te.shape[0])
                X_test.append(te)                        
            X_test = np.concatenate(X_test, axis=0)                   

            scaler = StandardScaler()
            X_train = scaler.fit_transform(X_train)
            X_test = scaler.transform(X_test)

            #X_train_selected = selector.fit_transform(X_train, y_train)
            #X_test_selected = selector.transform(X_test)
            X_train_selected = X_train
            X_test_selected = X_test

            # Train the classifier on the training data
            clf.fit(X_train_selected, y_train)

            # Make predictions on the test data
            y_pred = clf.predict(X_test_selected)

            # Calculate metrics for the current fold and append to the scores list
            accuracy_scores.append(accuracy_score(y_true=y_test, y_pred=y_pred))
            f1_scores.append(f1_score(y_true=y_test, y_pred=y_pred, average="macro"))
            auc_scores.append(roc_auc_score(y_test, y_pred, multi_class="ovr"))
            
            tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
            sen = tp / (tp + fn) if (tp + fn) != 0 else 0
            spe = tn / (fp + tn) if (fp + tn) != 0 else 0
            ppv = tp / (tp + fp) if (tp + fp) != 0 else 0
            npv = tn / (tn + fn) if (tn + fn) != 0 else 0

            sen_scores.append(sen)
            spe_scores.append(spe)
            ppv_scores.append(ppv)
            npv_scores.append(npv)

            count_num = 0
            vote_pred = []
            vote_test = []
            for v in vote_num:
                count = Counter(y_pred[count_num:count_num+v])
                most_common_value = count.most_common(1)[0][0]
                vote_pred.append(most_common_value)

                count = Counter(y_test[count_num:count_num+v])
                most_common_value = count.most_common(1)[0][0]
                vote_test.append(most_common_value)

                count_num = count_num + v

            vote_accuracy_scores.append(accuracy_score(y_true=vote_test, y_pred=vote_pred))
            vote_f1_scores.append(f1_score(y_true=vote_test, y_pred=vote_pred, average="macro"))
            vote_auc_scores.append(roc_auc_score(vote_test, vote_pred, multi_class="ovr"))

            vote_tn, vote_fp, vote_fn, vote_tp = confusion_matrix(vote_test, vote_pred).ravel()
            vote_sen = vote_tp / (vote_tp + vote_fn) if (vote_tp + vote_fn) != 0 else 0
            vote_spe = vote_tn / (vote_fp + vote_tn) if (vote_fp + vote_tn) != 0 else 0
            vote_ppv = vote_tp / (vote_tp + vote_fp) if (vote_tp + vote_fp) != 0 else 0
            vote_npv = vote_tn / (vote_tn + vote_fn) if (vote_tn + vote_fn) != 0 else 0

            vote_sen_scores.append(vote_sen)
            vote_spe_scores.append(vote_spe)
            vote_ppv_scores.append(vote_ppv)
            vote_npv_scores.append(vote_npv)

            vote_pred_list.append(vote_pred)
            vote_test_list.append(vote_test)
            pred_list.append(y_pred)
            test_list.append(y_test)

            # Save results for each option
            result = {
                'time': time,
                'classifier': clf_name,
                'feature-selection': selector_nm,
                'accuracy': np.mean(accuracy_scores),
                'f1_score': np.mean(f1_scores),
                'AUC': np.mean(auc_scores),
                'sen_scores': np.mean(sen_scores),
                'spe_scores': np.mean(spe_scores),
                'ppv_scores': np.mean(ppv_scores),
                'npv_scores': np.mean(npv_scores),
                'vote_accuracy': np.mean(vote_accuracy_scores),
                'vote_f1_score': np.mean(vote_f1_scores),
                'vote_AUC': np.mean(vote_auc_scores),
                'vote_sen_scores': np.mean(vote_sen_scores),
                'vote_spe_scores': np.mean(vote_spe_scores),
                'vote_ppv_scores': np.mean(vote_ppv_scores),
                'vote_npv_scores': np.mean(vote_npv_scores),
                'std_accuracy': np.std(accuracy_scores),
                'std_f1_score': np.std(f1_scores),
                'std_AUC': np.std(auc_scores),
                'std_sen_scores': np.std(sen_scores),
                'std_spe_scores': np.std(spe_scores),
                'std_ppv_scores': np.std(ppv_scores),
                'std_npv_scores': np.std(npv_scores),
                'std_vote_accuracy': np.std(vote_accuracy_scores),
                'std_vote_f1_score': np.std(vote_f1_scores),
                'std_vote_AUC': np.std(vote_auc_scores),
                'std_vote_sen_scores': np.std(vote_sen_scores),
                'std_vote_spe_scores': np.std(vote_spe_scores),
                'std_vote_ppv_scores': np.std(vote_ppv_scores),
                'std_vote_npv_scores': np.std(vote_npv_scores),
                'test': test_list, 
                'pred': pred_list,
                'vote_test': vote_test_list, 
                'vote_pred': vote_pred_list,
                #'selected_features': selector.get_support(indices=True)  # 선택된 특징 인덱스 추가
            }

            results.append(result)
    df_results = pd.DataFrame(results)
    return df_results#, clf
#여기
#분류
total_total_df_AD_CN = pd.DataFrame()
total_total_df_FTD_CN = pd.DataFrame()
total_total_df_AD_FTD_CN = pd.DataFrame()
total_total_df_AD_FTD = pd.DataFrame()



for r in random_list:
#여기

    total_df_AD_CN = pd.DataFrame()
    total_df_FTD_CN = pd.DataFrame()
    total_df_AD_FTD = pd.DataFrame()
    total_df_AD_FTD_CN = pd.DataFrame()

    results = []
    random.seed(r)
    for k in data_segments.keys():
        print('time: ',k)


        AD_total=0
        for i in range(len(a)):
            n = np.load("D:/test5/raw"+str(i+1)+"_AD_"+k+".npy").shape[0]
            AD_total+= n
        su=0
        AD_te = []
        numbers = list(range(0, len(a)))
        selected_indices = set()
        while su < AD_total*0.1:
            r = random.randint(0,len(a)-1)
            if r not in selected_indices:
                n = np.load("D:/test5/raw" + str(r+1) + "_AD_" + k + ".npy")
                su += n.shape[0]
                AD_te.append(r)
                selected_indices.add(r)
        AD_te.pop()
        AD_tr = [num for num in numbers if num not in AD_te]

        FTD_total=0
        for i in range(len(t)):
            n = np.load("D:/test5/raw"+str(i+66)+"_FTD_"+k+".npy").shape[0]
            FTD_total+= n    
        su=0
        FTD_te = []
        numbers = list(range(len(a)+len(c)+1, len(a)+len(c)+len(t)+1))
        selected_indices = set()
        while su < FTD_total*0.1:
            r = random.randint(len(a)+len(c)+1,len(a)+len(c)+len(t))
            if r not in selected_indices:
                n = np.load("D:/test5/raw" + str(r) + "_FTD_" + k + ".npy")
                su += n.shape[0]
                FTD_te.append(r)
                selected_indices.add(r)

        FTD_te.pop()
        FTD_tr = [num for num in numbers if num not in FTD_te]

        CN_total=0
        for i in range(len(c)):
            n = np.load("D:/test5/raw"+str(i+37)+"_CN_"+k+".npy").shape[0]
            CN_total+= n   
        su=0
        CN_te = []
        numbers = list(range(len(a)+1, len(a)+len(c)+1))
        selected_indices = set()
        while su < CN_total*0.1:
            r = random.randint(len(a)+1,len(a)+len(c))
            if r not in selected_indices:
                n = np.load("D:/test5/raw" + str(r) + "_CN_" + k + ".npy")
                su += n.shape[0]
                CN_te.append(r)
                selected_indices.add(r)    
        CN_te.pop()
        CN_tr = [num for num in numbers if num not in CN_te]


        #AD_FTD = np.concatenate((AD_feature, FTD_feature))
        #AD_CN = np.concatenate((np.load("C:/total/total/AD_total_+"+k+".npy"), np.load("C:/total/total/CN_total_+"+k+".npy")))
        #FTD_CN = np.concatenate((np.load("C:/total/total/FTD_total_+"+k+".npy"), np.load("C:/total/total/CN_total_+"+k+".npy")))
        #np.save("C:/total/feature/AD_CN_"+k, AD_CN)
        #np.save("C:/total/feature/FTD_CN_"+k, FTD_CN)



        AD_CN_tr = AD_tr+CN_tr
        AD_CN_te = AD_te+CN_te

        FTD_CN_tr = FTD_tr+CN_tr
        FTD_CN_te = FTD_te+CN_te

        AD_FTD_CN_tr = AD_tr+FTD_tr+CN_tr
        AD_FTD_CN_te = AD_te+FTD_te+CN_te
        
     
        


        #random.shuffle(AD_CN_tr)
        #random.shuffle(AD_CN_te)

        #random.shuffle(FTD_CN_tr)
        #random.shuffle(FTD_CN_te)

        #random.shuffle(AD_FTD_CN_tr)
        #random.shuffle(AD_FTD_CN_te)

   


        df_results_cv_allch_AD_CN = eeg_classifier_cv(k, ad_cn, AD_CN_tr, AD_CN_te)

        total_df_AD_CN = pd.concat((total_df_AD_CN,df_results_cv_allch_AD_CN))



        df_results_cv_allch_FTD_CN = eeg_classifier_cv(k, ftd_cn, FTD_CN_tr, FTD_CN_te)


        total_df_FTD_CN = pd.concat((total_df_FTD_CN,df_results_cv_allch_FTD_CN))



        df_results_cv_allch_AD_FTD_CN = eeg_classifier_cv(k, ad_ftd_cn, AD_FTD_CN_tr, AD_FTD_CN_te)

        total_df_AD_FTD_CN = pd.concat((total_df_AD_FTD_CN,df_results_cv_allch_AD_FTD_CN))


        
        

    total_total_df_AD_CN = pd.concat((total_total_df_AD_CN,total_df_AD_CN))

    total_total_df_FTD_CN = pd.concat((total_total_df_FTD_CN,total_df_FTD_CN))

    total_total_df_AD_FTD_CN = pd.concat((total_total_df_AD_FTD_CN,total_df_AD_FTD_CN))